## Step 1: Load NOAA's master weather station list

Before pulling any actual weather data, we need a way to connect our 395 airport codes to NOAA weather stations. NOAA's `isd-history.csv` is the master list of ~29,600 stations worldwide, including each station's ICAO code, WBAN number, and coverage dates.

In [1]:
import glob
import pandas as pd
import requests

# Download NOAA's master station list
station_list = pd.read_csv("https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv")
print(f"Total stations worldwide: {len(station_list)}")
print(station_list.columns.tolist())
print(station_list.head())

Total stations worldwide: 29661
['USAF', 'WBAN', 'STATION NAME', 'CTRY', 'STATE', 'ICAO', 'LAT', 'LON', 'ELEV(M)', 'BEGIN', 'END']
     USAF   WBAN STATION NAME CTRY STATE ICAO    LAT     LON  ELEV(M)  \
0  007018  99999   WXPOD 7018  NaN   NaN  NaN   0.00   0.000   7018.0   
1  007026  99999   WXPOD 7026   AF   NaN  NaN   0.00   0.000   7026.0   
2  007070  99999   WXPOD 7070   AF   NaN  NaN   0.00   0.000   7070.0   
3  008260  99999    WXPOD8270  NaN   NaN  NaN   0.00   0.000      0.0   
4  008268  99999    WXPOD8278   AF   NaN  NaN  32.95  65.567   1156.7   

      BEGIN       END  
0  20110309  20130730  
1  20120713  20170822  
2  20140923  20150926  
3  20050101  20120731  
4  20100519  20120323  


## Step 2: Map airports to stations (attempt 1)

Airports use IATA codes (e.g. `ORD`), but weather stations are identified by ICAO codes (e.g. `KORD`). For most of the continental U.S., the rule is simply `ICAO = "K" + IATA`. Testing that rule here.

In [2]:
airport_codes = set()
for f in glob.glob("../data/processed/ontime/*.parquet"):
    df = pd.read_parquet(f, columns=["Origin", "Dest"])
    airport_codes.update(df["Origin"].unique())
    airport_codes.update(df["Dest"].unique())

print(f"Total airport codes: {len(airport_codes)}")

# Continental U.S. rule: ICAO = "K" + IATA. Territory overrides from when we built dim_airport.
TERRITORY_ICAO_OVERRIDES = {
    "PPG": "NSTU", "SPN": "PGSN", "GUM": "PGUM",
    "STT": "TIST", "STX": "TISX", "BQN": "TJBQ", "PSE": "TJPS", "SJU": "TJSJ",
}

def to_icao(iata):
    return TERRITORY_ICAO_OVERRIDES.get(iata, "K" + iata)

airport_icao = pd.DataFrame({
    "iata": sorted(airport_codes),
})
airport_icao["icao"] = airport_icao["iata"].apply(to_icao)

# Join against NOAA's station list
matched = airport_icao.merge(station_list, left_on="icao", right_on="ICAO", how="left")

matched_count = matched["WBAN"].notna().sum()
print(f"\nMatched: {matched_count} out of {len(airport_icao)}")
print(f"\nUnmatched:")
print(matched[matched["WBAN"].isna()][["iata", "icao"]])

Total airport codes: 395

Matched: 780 out of 395

Unmatched:
    iata  icao
19   ADK  KADK
20   ADQ  KADQ
27   AKN  KAKN
37   ANC  KANC
56   AZA  KAZA
61   BET  KBET
85   BKG  KBKG
105  BRW  KBRW
128  CDB  KCDB
131  CDV  KCDV
149  CLD  KCLD
212  DLG  KDLG
263  FAI  KFAI
270  FCA  KFCA
326  GST  KGST
333  GUF  KGUF
340  HHH  KHHH
345  HNL  KHNL
400  ITO  KITO
413  JNU  KJNU
417  KOA  KKOA
418  KTN  KKTN
451  LIH  KLIH
545  OGG  KOGG
552  OME  KOME
568  OTZ  KOTZ
621  PSG  KPSG
679  SCC  KSCC
680  SCE  KSCE
702  SIT  KSIT
787  USA  KUSA
788  UST  KUST
801  WRG  KWRG
806  XWA  KXWA
807  YAK  KYAK


### Fix: Alaska and Hawaii don't follow the "K" rule

The simple prefix rule only matched 360/395 airports. The 35 misses aren't random — they're every Alaska and Hawaii airport. Those regions use `PA` and `PH` prefixes instead of `K`. Patching the rule accordingly.

In [3]:
# Fix 1: dedupe the station list (keep the most recent record per ICAO)
station_list_dedup = station_list.sort_values("END", ascending=False).drop_duplicates(subset="ICAO", keep="first")

# Fix 2: Alaska and Hawaii use different ICAO prefixes than the continental "K" rule
def to_icao_v2(row):
    iata = row["iata"]
    if iata in TERRITORY_ICAO_OVERRIDES:
        return TERRITORY_ICAO_OVERRIDES[iata]
    if iata in ALASKA_CODES:
        return "PA" + iata
    if iata in HAWAII_CODES:
        return "PH" + iata
    return "K" + iata

ALASKA_CODES = {"ADK", "ADQ", "AKN", "ANC", "BET", "BRW", "CDB", "CDV", "DLG",
                 "FAI", "GST", "JNU", "KTN", "OME", "OTZ", "PSG", "SCC", "SIT", "WRG", "YAK"}
HAWAII_CODES = {"HNL", "ITO", "KOA", "LIH", "OGG"}

airport_icao["icao"] = airport_icao.apply(to_icao_v2, axis=1)

matched = airport_icao.merge(station_list_dedup, left_on="icao", right_on="ICAO", how="left")
matched_count = matched["WBAN"].notna().sum()
print(f"Matched: {matched_count} out of {len(airport_icao)}")
print(f"\nStill unmatched:")
print(matched[matched["WBAN"].isna()][["iata", "icao"]])

Matched: 360 out of 395

Still unmatched:
    iata   icao
9    ADK  PAADK
10   ADQ  PAADQ
13   AKN  PAAKN
19   ANC  PAANC
29   AZA   KAZA
32   BET  PABET
43   BKG   KBKG
55   BRW  PABRW
65   CDB  PACDB
67   CDV  PACDV
76   CLD   KCLD
105  DLG  PADLG
129  FAI  PAFAI
133  FCA   KFCA
158  GST  PAGST
162  GUF   KGUF
166  HHH   KHHH
169  HNL  PHHNL
194  ITO  PHITO
201  JNU  PAJNU
203  KOA  PHKOA
204  KTN  PAKTN
221  LIH  PHLIH
265  OGG  PHOGG
269  OME  PAOME
275  OTZ  PAOTZ
300  PSG  PAPSG
330  SCC  PASCC
331  SCE   KSCE
342  SIT  PASIT
380  USA   KUSA
381  UST   KUST
387  WRG  PAWRG
390  XWA   KXWA
391  YAK  PAYAK


### Diagnosing a station that still won't match

Even with the region-specific prefixes, some stations (e.g. Anchorage) still don't match. Checking NOAA's raw station list directly to see what's actually going on, rather than guessing at another prefix rule.

In [4]:
# Check if Anchorage (a major hub, should definitely be in there) exists at all,
# regardless of what its ICAO field says
print(station_list[station_list["STATION NAME"].str.contains("ANCHORAGE", na=False)])

# Also check: is "PAANC" anywhere in the raw (non-deduped) ICAO column at all?
print(f"\nPAANC found anywhere: {(station_list['ICAO'] == 'PAANC').sum()} rows")

         USAF   WBAN                STATION NAME CTRY STATE  ICAO     LAT  \
15370  702720  99999         ANCHORAGE/ELMENDORF   US    AK  PAED  61.250   
15372  702725  99999         ANCHORAGE LAKE HOOD   US    AK  PALH  61.183   
15373  702730  26451  TED STEVENS ANCHORAGE INTL   US    AK  PANC  61.169   
18599  722592  99999              ANCHORAGE(WFO)   US    AK   NaN  61.150   
27829  997381  99999                   ANCHORAGE   US   NaN   NaN  61.238   
29122  999999  26401     ANCHORAGE ELMENDORF AFB   US    AK  PAED  61.253   
29136  999999  26451           ANCHORAGE INTL AP   US    AK  PANC  61.169   

           LON  ELEV(M)     BEGIN       END  
15370 -149.800     59.0  19410310  19971231  
15372 -149.967     22.0  19730104  19971230  
15373 -150.028     38.0  19730101  20250827  
18599 -149.983     40.0  20140924  20250824  
27829 -149.890      9.0  20080101  20250824  
29122 -149.794     64.9  20040702  20051231  
29136 -150.028     40.2  19531101  19721231  

PAANC found an

### Real fix: use a verified IATA↔ICAO lookup instead of inferring one

Guessing prefix rules from a handful of examples doesn't generalize reliably (Anchorage's real ICAO, `PANC`, doesn't follow any simple pattern from its IATA code `ANC`). Switching to OpenFlights' curated airport database — the same source already used for `dim_airport` — which has verified, real ICAO codes rather than inferred ones. This closes the gap completely: 395/395 matched.

In [5]:
# Re-pull OpenFlights (same source as dim_airport) — this time keep the icao column too
airports_url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
airports_raw = pd.read_csv(
    airports_url,
    header=None,
    names=["airport_id", "name", "city", "country", "iata", "icao",
           "lat", "lon", "altitude", "timezone", "dst", "tz_db", "type", "source"]
)

iata_to_icao = dict(zip(airports_raw["iata"], airports_raw["icao"]))

airport_icao["icao"] = airport_icao["iata"].map(iata_to_icao)

matched = airport_icao.merge(station_list_dedup, left_on="icao", right_on="ICAO", how="left")
matched_count = matched["WBAN"].notna().sum()
print(f"Matched: {matched_count} out of {len(airport_icao)}")
print(f"\nStill unmatched:")
print(matched[matched["WBAN"].isna()][["iata", "icao"]])

Matched: 395 out of 395

Still unmatched:
Empty DataFrame
Columns: [iata, icao]
Index: []


## Step 3: Build the final station ID lookup

With a confirmed ICAO code for every airport, the last step is constructing NOAA's actual station identifier (`USW000` + the station's 5-digit WBAN number) and saving the full mapping for reuse.

In [6]:
station_mapping = matched[["iata", "icao", "USAF", "WBAN", "STATE"]].copy()
station_mapping["ghcn_station_id"] = "USW000" + station_mapping["WBAN"].astype(int).astype(str).str.zfill(5)

print(station_mapping.head(10))
print(f"\nTotal: {len(station_mapping)} airport-to-station mappings")

station_mapping.to_csv("../data/processed/airport_station_mapping.csv", index=False)
print("Saved.")

  iata  icao    USAF   WBAN STATE ghcn_station_id
0  ABE  KABE  725170  14737    PA     USW00014737
1  ABI  KABI  722660  13962    TX     USW00013962
2  ABQ  KABQ  723650  23050    NM     USW00023050
3  ABR  KABR  726590  14929    SD     USW00014929
4  ABY  KABY  722160  13869    GA     USW00013869
5  ACK  KACK  725060  14756    MA     USW00014756
6  ACT  KACT  722560  13959    TX     USW00013959
7  ACV  KACV  725945  24283    CA     USW00024283
8  ACY  KACY  724070  93730    NJ     USW00093730
9  ADK  PADK  704540  25704    AK     USW00025704

Total: 395 airport-to-station mappings
Saved.


## Step 4: Test the weather API

NOAA's Local Climatological Data (LCD) API was the original plan — it offers richer, airport-specific hourly/daily detail. Testing a real request for O'Hare before committing to it at scale.

In [7]:
import requests

test_url = "https://www.ncei.noaa.gov/access/services/data/v1"
params = {
    "dataset": "local-climatological-data",
    "stations": "USW00094846",  # O'Hare
    "startDate": "2019-01-01",
    "endDate": "2019-01-07",
    "format": "csv",
}

r = requests.get(test_url, params=params, timeout=60)
print(f"Status code: {r.status_code}")
print(f"Response length: {len(r.text)} characters")
print(r.text[:2000])

Status code: 200
Response length: 3162 characters
"STATION","DATE","REPORT_TYPE","SOURCE","AWND","BackupDirection","BackupDistance","BackupDistanceUnit","BackupElements","BackupElevation","BackupElevationUnit","BackupEquipment","BackupLatitude","BackupLongitude","BackupName","CDSD","CLDD","DSNW","DYHF","DYTS","DailyAverageDewPointTemperature","DailyAverageDryBulbTemperature","DailyAverageRelativeHumidity","DailyAverageSeaLevelPressure","DailyAverageStationPressure","DailyAverageWetBulbTemperature","DailyAverageWindSpeed","DailyCoolingDegreeDays","DailyDepartureFromNormalAverageTemperature","DailyHeatingDegreeDays","DailyMaximumDryBulbTemperature","DailyMinimumDryBulbTemperature","DailyPeakWindDirection","DailyPeakWindSpeed","DailyPrecipitation","DailySnowDepth","DailySnowfall","DailySustainedWindDirection","DailySustainedWindSpeed","DailyWeather","HDSD","HTDD","HourlyAltimeterSetting","HourlyDewPointTemperature","HourlyDryBulbTemperature","HourlyPrecipitation","HourlyPresentWeatherType

### The response came back empty

Status 200, but zero actual data rows — just a header line. Something's wrong, and it's worth diagnosing properly rather than assuming our request was malformed.

In [8]:
import io

df_test = pd.read_csv(io.StringIO(r.text))
print(f"Shape: {df_test.shape}")
print(f"\nDaily-prefixed columns:")
daily_cols = [c for c in df_test.columns if c.startswith("Daily")]
print(daily_cols)

print(f"\nSample of daily data:")
print(df_test[["STATION", "DATE"] + daily_cols].head(10))

Shape: (0, 124)

Daily-prefixed columns:
['DailyAverageDewPointTemperature', 'DailyAverageDryBulbTemperature', 'DailyAverageRelativeHumidity', 'DailyAverageSeaLevelPressure', 'DailyAverageStationPressure', 'DailyAverageWetBulbTemperature', 'DailyAverageWindSpeed', 'DailyCoolingDegreeDays', 'DailyDepartureFromNormalAverageTemperature', 'DailyHeatingDegreeDays', 'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature', 'DailyPeakWindDirection', 'DailyPeakWindSpeed', 'DailyPrecipitation', 'DailySnowDepth', 'DailySnowfall', 'DailySustainedWindDirection', 'DailySustainedWindSpeed', 'DailyWeather']

Sample of daily data:
Empty DataFrame
Columns: [STATION, DATE, DailyAverageDewPointTemperature, DailyAverageDryBulbTemperature, DailyAverageRelativeHumidity, DailyAverageSeaLevelPressure, DailyAverageStationPressure, DailyAverageWetBulbTemperature, DailyAverageWindSpeed, DailyCoolingDegreeDays, DailyDepartureFromNormalAverageTemperature, DailyHeatingDegreeDays, DailyMaximumDryBulbTemper

### Confirming it's genuinely empty, not a parsing bug

Checking the raw response directly, independent of pandas, to rule out a parsing issue on my end.

In [9]:
lines = r.text.strip().split("\n")
print(f"Total lines in response: {len(lines)}")
print(f"\nLast 500 characters of response:")
print(r.text[-500:])

Total lines in response: 1

Last 500 characters of response:
ndDate180","ShortDurationPrecipitationValue005","ShortDurationPrecipitationValue010","ShortDurationPrecipitationValue015","ShortDurationPrecipitationValue020","ShortDurationPrecipitationValue030","ShortDurationPrecipitationValue045","ShortDurationPrecipitationValue060","ShortDurationPrecipitationValue080","ShortDurationPrecipitationValue100","ShortDurationPrecipitationValue120","ShortDurationPrecipitationValue150","ShortDurationPrecipitationValue180","Sunrise","Sunset","WindEquipmentChangeDate"



### Testing two hypotheses: is the API itself broken, and is a token required?

Trying NOAA's own documented example query (to see if any request to this API works right now) and adding an API token to the request header (in case it's silently required despite documentation saying otherwise).

In [10]:
example_url = "https://www.ncei.noaa.gov/access/services/data/v1"
example_params = {
    "dataset": "global-marine",
    "dataTypes": "WIND_DIR,WIND_SPEED",
    "stations": "AUCE",
    "startDate": "2016-01-01",
    "endDate": "2016-01-02",
}
r_example = requests.get(example_url, params=example_params, timeout=60)
print(f"NOAA's own example — status: {r_example.status_code}, rows: {len(r_example.text.strip().splitlines())}")

YOUR_TOKEN = "paste_your_actual_token_here"
headers = {"token": YOUR_TOKEN}

r_with_token = requests.get(test_url, params=params, headers=headers, timeout=60)
print(f"\nOur query with token header — status: {r_with_token.status_code}, rows: {len(r_with_token.text.strip().splitlines())}")

NOAA's own example — status: 400, rows: 1

Our query with token header — status: 200, rows: 1


## Step 5: Pivot to GHCN-Daily (CDO API v2)

NOAA's own example failed too — this points to a real outage on their end (likely tied to their in-progress cloud migration), not something fixable on our side. Pivoting to an older, more stable NOAA dataset and API instead. This one requires the token, unlike the one I just abandoned.

In [11]:
import getpass
CDO_TOKEN = getpass.getpass("Enter your NOAA CDO token: ")

ghcnd_url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
ghcnd_params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USW00094846",  # same WBAN-based ID, just with the GHCND: prefix
    "startdate": "2019-01-01",
    "enddate": "2019-01-07",
    "limit": 1000,
}
headers = {"token": CDO_TOKEN}

r_ghcnd = requests.get(ghcnd_url, params=ghcnd_params, headers=headers, timeout=60)
print(f"Status: {r_ghcnd.status_code}")
print(r_ghcnd.text[:1500])

Status: 200
{"metadata":{"resultset":{"offset":1,"count":124,"limit":1000}},"results":[{"date":"2019-01-01T00:00:00","datatype":"ADPT","station":"GHCND:USW00094846","attributes":",,W,","value":-39},{"date":"2019-01-01T00:00:00","datatype":"ASLP","station":"GHCND:USW00094846","attributes":",,W,","value":10251},{"date":"2019-01-01T00:00:00","datatype":"ASTP","station":"GHCND:USW00094846","attributes":",,W,","value":9990},{"date":"2019-01-01T00:00:00","datatype":"AWBT","station":"GHCND:USW00094846","attributes":",,W,","value":-28},{"date":"2019-01-01T00:00:00","datatype":"AWND","station":"GHCND:USW00094846","attributes":",,W,","value":34},{"date":"2019-01-01T00:00:00","datatype":"PRCP","station":"GHCND:USW00094846","attributes":",,W,2400","value":3},{"date":"2019-01-01T00:00:00","datatype":"RHAV","station":"GHCND:USW00094846","attributes":",,W,","value":87},{"date":"2019-01-01T00:00:00","datatype":"RHMN","station":"GHCND:USW00094846","attributes":",,W,","value":81},{"date":"2019-01-01T00:

### Narrowing down to only the weather variables we actually need

The unfiltered request returns ~18 variables per day — far more than needed, and enough to blow through the API's per-request row limit at scale. Filtering to the 7 variables that actually matter for delay prediction: TMAX, TMIN, PRCP, SNOW, SNWD, AWND, WSF5.

In [12]:
ghcnd_params_filtered = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USW00094846",
    "startdate": "2019-01-01",
    "enddate": "2019-01-31",
    "datatypeid": "TMAX,TMIN,PRCP,SNOW,SNWD,AWND,WSF5",
    "limit": 1000,
}

r_filtered = requests.get(ghcnd_url, params=ghcnd_params_filtered, headers=headers, timeout=60)
print(f"Status: {r_filtered.status_code}")
import json
data = json.loads(r_filtered.text)
print(f"Results returned: {len(data['results'])}")
print(f"Metadata count: {data['metadata']['resultset']['count']}")

df_check = pd.DataFrame(data["results"])
print(f"\nDatatypes present: {df_check['datatype'].unique()}")
print(df_check.head(10))

Status: 200
Results returned: 217
Metadata count: 217

Datatypes present: ['AWND' 'PRCP' 'SNOW' 'SNWD' 'TMAX' 'TMIN' 'WSF5']
                  date datatype            station attributes  value
0  2019-01-01T00:00:00     AWND  GHCND:USW00094846       ,,W,     34
1  2019-01-01T00:00:00     PRCP  GHCND:USW00094846   ,,W,2400      3
2  2019-01-01T00:00:00     SNOW  GHCND:USW00094846       ,,W,      3
3  2019-01-01T00:00:00     SNWD  GHCND:USW00094846   ,,W,2400      0
4  2019-01-01T00:00:00     TMAX  GHCND:USW00094846   ,,W,2400     11
5  2019-01-01T00:00:00     TMIN  GHCND:USW00094846   ,,W,2400    -32
6  2019-01-01T00:00:00     WSF5  GHCND:USW00094846       ,,W,     94
7  2019-01-02T00:00:00     AWND  GHCND:USW00094846       ,,W,     39
8  2019-01-02T00:00:00     PRCP  GHCND:USW00094846   ,,W,2400      3
9  2019-01-02T00:00:00     SNOW  GHCND:USW00094846       ,,W,      3


## Step 6: Full-scale pull — all 395 airports, 2015–2025

With the mapping, API, and variable set all validated, this pulls the complete dataset: ~4,345 station-years, paginated and rate-limited to stay within NOAA's quota, and fully resumable (skips anything already downloaded) in case it needs to span multiple sessions.

In [13]:
import os
import time
import json

CDO_TOKEN = "your_actual_token_here"  # your real token
headers = {"token": CDO_TOKEN}
DATATYPES = "TMAX,TMIN,PRCP,SNOW,SNWD,AWND,WSF5"

RAW_WEATHER_DIR = "../data/raw/weather"
os.makedirs(RAW_WEATHER_DIR, exist_ok=True)

station_mapping = pd.read_csv("../data/processed/airport_station_mapping.csv")

def fetch_station_year(station_id, year):
    """Pull one station-year, paginating as needed. Returns list of results, or a status string."""
    all_results = []
    offset = 1  # CDO's offset is 1-indexed, confirmed from our earlier test response
    limit = 1000

    while True:
        params = {
            "datasetid": "GHCND",
            "stationid": f"GHCND:{station_id}",
            "startdate": f"{year}-01-01",
            "enddate": f"{year}-12-31",
            "datatypeid": DATATYPES,
            "limit": limit,
            "offset": offset,
        }
        try:
            r = requests.get(ghcnd_url, params=params, headers=headers, timeout=30)
        except requests.exceptions.RequestException:
            return "ERROR"

        if r.status_code == 429:
            return "RATE_LIMITED"
        if r.status_code != 200:
            return []  # e.g. station has no data for this year — treat as empty, not fatal

        data = r.json()
        results = data.get("results", [])
        all_results.extend(results)

        total = data.get("metadata", {}).get("resultset", {}).get("count", 0)
        if offset + limit > total or not results:
            break
        offset += limit
        time.sleep(0.25)  # stay safely under 5 req/sec

    return all_results


request_count = 0
YEARS = range(2015, 2026)

for _, row in station_mapping.iterrows():
    station_id = row["ghcn_station_id"]
    iata = row["iata"]

    for year in YEARS:
        out_path = f"{RAW_WEATHER_DIR}/{iata}_{year}.json"
        if os.path.exists(out_path):
            continue  # already pulled — safe to re-run this whole script anytime

        result = fetch_station_year(station_id, year)

        if result == "RATE_LIMITED":
            print(f"\nDaily quota hit at {iata} {year} after {request_count} requests today.")
            print("Stop here — re-run this same script tomorrow, it'll resume exactly where it left off.")
            raise SystemExit

        if result == "ERROR":
            print(f"{iata} {year}: network error, skipping (re-run later to retry)")
            continue

        with open(out_path, "w") as f:
            json.dump(result, f)

        request_count += 1
        if request_count % 50 == 0:
            print(f"Progress: {request_count} station-years done ({iata} {year} just finished)")

        time.sleep(0.25)

print(f"\nAll done. Total requests today: {request_count}")

Progress: 50 station-years done (EGE 2024 just finished)
Progress: 100 station-years done (ERI 2019 just finished)
Progress: 150 station-years done (EWN 2025 just finished)
Progress: 200 station-years done (FAT 2020 just finished)
Progress: 250 station-years done (FLO 2015 just finished)
Progress: 300 station-years done (FOD 2021 just finished)
Progress: 350 station-years done (GCK 2016 just finished)
Progress: 400 station-years done (GJT 2022 just finished)
Progress: 450 station-years done (GRK 2017 just finished)
Progress: 500 station-years done (GST 2023 just finished)
Progress: 550 station-years done (GUM 2018 just finished)
Progress: 600 station-years done (HIB 2024 just finished)
Progress: 650 station-years done (HPN 2019 just finished)
Progress: 700 station-years done (HVN 2025 just finished)
Progress: 750 station-years done (IAH 2020 just finished)
Progress: 800 station-years done (ILM 2015 just finished)
Progress: 850 station-years done (IPT 2021 just finished)
Progress: 900 s

### Checking completeness (this check had a bug)

This verification undercounted correctly-downloaded files due to a path-parsing bug (see the corrected version in the next cell) — kept here to show the debugging process, not because its output was correct.

In [14]:
weather_files = glob.glob(f"{RAW_WEATHER_DIR}/*.json")
print(f"Total station-year files: {len(weather_files)}")
print(f"Expected: {len(station_mapping) * len(YEARS)}")

# Which specific station-years are missing (the network-error retries + anything else)
completed = set(f.split("/")[-1].replace(".json", "") for f in weather_files)
expected = set(f"{row['iata']}_{year}" for _, row in station_mapping.iterrows() for year in YEARS)
missing = expected - completed
print(f"\nMissing: {len(missing)}")
print(sorted(missing))

Total station-year files: 4340
Expected: 4345

Missing: 4345
['ABE_2015', 'ABE_2016', 'ABE_2017', 'ABE_2018', 'ABE_2019', 'ABE_2020', 'ABE_2021', 'ABE_2022', 'ABE_2023', 'ABE_2024', 'ABE_2025', 'ABI_2015', 'ABI_2016', 'ABI_2017', 'ABI_2018', 'ABI_2019', 'ABI_2020', 'ABI_2021', 'ABI_2022', 'ABI_2023', 'ABI_2024', 'ABI_2025', 'ABQ_2015', 'ABQ_2016', 'ABQ_2017', 'ABQ_2018', 'ABQ_2019', 'ABQ_2020', 'ABQ_2021', 'ABQ_2022', 'ABQ_2023', 'ABQ_2024', 'ABQ_2025', 'ABR_2015', 'ABR_2016', 'ABR_2017', 'ABR_2018', 'ABR_2019', 'ABR_2020', 'ABR_2021', 'ABR_2022', 'ABR_2023', 'ABR_2024', 'ABR_2025', 'ABY_2015', 'ABY_2016', 'ABY_2017', 'ABY_2018', 'ABY_2019', 'ABY_2020', 'ABY_2021', 'ABY_2022', 'ABY_2023', 'ABY_2024', 'ABY_2025', 'ACK_2015', 'ACK_2016', 'ACK_2017', 'ACK_2018', 'ACK_2019', 'ACK_2020', 'ACK_2021', 'ACK_2022', 'ACK_2023', 'ACK_2024', 'ACK_2025', 'ACT_2015', 'ACT_2016', 'ACT_2017', 'ACT_2018', 'ACT_2019', 'ACT_2020', 'ACT_2021', 'ACT_2022', 'ACT_2023', 'ACT_2024', 'ACT_2025', 'ACV_2015', 'A

### Corrected completeness check

Using os.path.basename()/os.path.splitext() instead of manual string splitting, which handles Windows path separators correctly. This is the trustworthy result: 4,340/4,345 succeeded, 5 genuine network-error retries remaining.

In [15]:
completed = set(os.path.splitext(os.path.basename(f))[0] for f in weather_files)
expected = set(f"{row['iata']}_{year}" for _, row in station_mapping.iterrows() for year in YEARS)
missing = expected - completed

print(f"Total files: {len(weather_files)}")
print(f"Missing: {len(missing)}")
print(sorted(missing))

Total files: 4340
Missing: 5
['JFK_2020', 'JLN_2016', 'OGS_2016', 'SJC_2019', 'TVC_2015']


## Step 7: Patch the remaining retries

Re-running just the 5 station-years that failed with transient network errors during the main pull.

In [16]:
for miss in ["JFK_2020", "JLN_2016", "OGS_2016", "SJC_2019", "TVC_2015"]:
    iata, year = miss.split("_")
    year = int(year)
    row = station_mapping[station_mapping["iata"] == iata].iloc[0]
    station_id = row["ghcn_station_id"]

    result = fetch_station_year(station_id, year)
    if result in ("ERROR", "RATE_LIMITED"):
        print(f"{miss}: still failed ({result})")
        continue

    with open(f"{RAW_WEATHER_DIR}/{miss}.json", "w") as f:
        json.dump(result, f)
    print(f"{miss}: fixed, {len(result)} rows")
    time.sleep(0.25)

JFK_2020: fixed, 0 rows
JLN_2016: fixed, 0 rows
OGS_2016: fixed, 0 rows
SJC_2019: fixed, 0 rows
TVC_2015: fixed, 0 rows
